# KataGo Human SL 分析

執行 `aicup.analyze_humansl`，輸出每局在各 profile 下的落子機率，寫成 jsonl。

- 外部資源：KataGo `v1.18.2`、Human SL 網路 `b18c384nbt-humanv0`、主網路 `kata1-b28c512nbt-s13255194368-d5935380940`，授權見 KataGo 官方 repo
- 規則、貼目、讓子依官方 Q&A，預設 `japanese`、6.5 目，`HD` 的處理見 `parse_game`
- 輸入與模型先複製到 `/content`，分析期間不讀 Drive；結果寫在 `/content`，每批結束後複製到 Drive
- 斷線時最多損失最近一批。重新開 runtime，依序執行全部 cell 會從 Drive 上的結果接續；分析中不要中斷 cell
- 6d 的 `*_hist` 與 `pub2` 沒有對照組檢查：沒有公開的參考輸出可比，KataGo 版本與網路檔由下載網址固定
- 2d 的 MODE 先做對照組檢查，見「對照組檢查」一節

競賽一最終版本用到的 MODE：

| MODE | 局 | 用途 |
|---|---|---|
| `trainpool_hist` | train 池 1,000 位玩家各 20 局 | B 組段位 LR 的擬合資料 |
| `val648_hist` | val648 的 648 位玩家各 20 局 | 同上 |
| `newdev_hist` | newdev 的 338 位玩家各 20 局 | 同上 |
| `pub2` | B 組 public 與 private 的候選局與題目局 | B 組段位估計的輸入 |

- 這 4 個 MODE 都評目標玩家的手，3 個 profile（`rank_11k`、`rank_1d`、`rank_6d`），帶 `--pre-root-history`，不算損失。
- `pub2_smoke` 只跑 `pub2` 的前 100 局。

其他 MODE 不在競賽一最終版本流程內：

- `trainpool_hist_opp`、`val648_hist_opp`、`newdev_hist_opp`、`pub2_opp`：去掉 `_opp` 的 MODE 的局，改評對手的手（`--side opponent`）。
- `val648`、`val648_opp`、`valnew`、`valnew_opp`、`trainpool`、`trainpool_opp`：競賽二的分析，從訓練 CSV 依玩家清單抽 20 局，不帶 `--pre-root-history`。
- `test`、`test_opp`：競賽二的 10,000 局測試題。
- `test_hist`、`test_hist_opp`：競賽二的測試題，設定同 `trainpool_hist`。
- `pilot`、`visits`、`main`：競賽二初期的試跑。
- `*_2d`、`*_2d_opp`：競賽二 Transformer 的 2d 資料，見下一節。
- `FAST`：只用於 `val648_opp`、`valnew_opp`、`test`、`test_opp`。主模型改用 Human SL 網路並加大 GPU 批次；不算損失時主模型的輸出用不到。分析前先用 val648 的 40 局和 `val648` 的輸出比對，不一致時改回原設定。

競賽二最終版本的 Transformer 用到的 MODE：

| MODE | 局 | 用途 |
|---|---|---|
| `trainpool_hist`、`newdev_hist`、`val648_hist`、`test_hist` 與各自的 `_opp` | 見上方 | 6d 池與 6d 的 val648、測試題 |
| `train_p1_2d`、`train_p2_2d`、`train_p3_2d` | 各 1,000 位玩家各 20 局 | 2d 池 |
| `train_p4_2d`、`train_p4b_2d` | 1,995 位玩家多數色 10 局，其中 1,198 位同色再 10 局 | 2d 池，打包時串接 |
| `val648_2d`、`test_2d` | val648、測試題 | 2d 的 val648 與測試題 |
| 以上 2d MODE 的 `_opp` | 同上 | 對手側 |

- 2d MODE 用 3 個 profile（`rank_11k`、`rank_1d`、`rank_2d`），帶 `--pre-root-history`、`--move-stats`，主網路算每手損失（`--eval-visits 1`），`numAnalysisThreads=32`。
- `train_p4_2d`、`train_p4b_2d` 不算損失（`--no-eval`）。
- `holdout_2d`、`holdout_2d_opp` 是封存組，不在最終版本的訓練資料內。

## 需要的檔案

路徑都在 Drive 的 `MyDrive/aicup-2026/` 下，在本機產生後上傳。KataGo 執行檔與網路檔由 notebook 從官方網址下載，不放 Drive：`b18c384nbt-humanv0.bin.gz` 來自 KataGo 的 GitHub Releases，主網路 `kata1-b28c512nbt-s13255194368-d5935380940.bin.gz` 來自 `media.katagotraining.org`，只在用主網路的 MODE 下載（名稱不含 `_hist` 的 MODE）。

| 路徑 | MODE | 產生方式（本機，repo 根目錄） |
|---|---|---|
| `humansl-analysis/hist6d-trainpool/games.csv` | `trainpool_hist*` | `python -m aicup.hsl_games --run trainpool` |
| `humansl-analysis/hist6d-val648/games.csv` | `val648_hist*` | `python -m aicup.hsl_games --run val648` |
| `humansl-analysis/hist6d-newdev/games.csv` | `newdev_hist*` | `python -m aicup.hsl_games --run newdev` |
| `humansl-analysis/hist2d-<run>/games.csv` | `<run>_2d*`（`test_2d*` 除外） | `python -m aicup.hsl_games --2d --run <run>` |
| `player-trained-models/hsl-cache/player_hsl_pub2_manifest.csv` | `pub2*` | `python -m aicup.prep_player_hsl --tests-dir tests` |
| `training-v1.1/training/train_*.csv` | `val648`、`val648_opp`、`valnew*`、`trainpool`、`trainpool_opp`、`pilot`、`visits`、`main`，以及 `FAST` 的比對 | 官方訓練集 v1.1，見 README |
| `humansl-analysis/val-players.csv` | `val648`、`val648_opp`，以及 `FAST` 的比對 | `python -m aicup.val_players --val-size 180000 --out-dir humansl-analysis` |
| `humansl-analysis/val-players-new-v180000.csv` | `valnew`、`valnew_opp` | 同上 |
| `humansl-analysis/train-pool-v180000.csv` | `trainpool`、`trainpool_opp` | 同上 |
| `tests/rank_prediction_test_{public,private}.csv` | `test*`（含 `test_hist*`、`test_2d*`） | AI CUP 競賽平台 |
| `humansl-analysis/val648-p3.jsonl` | `FAST` 的比對、2d MODE 的對照組 | 這份 notebook 的 `val648` |
| `humansl-analysis/hist2d-val648/games.csv` | 2d MODE 的對照組 | `python -m aicup.hsl_games --2d --run val648` |

## 參數

`BATCH` 是每批局數，每批結束後備份到 Drive。

In [ ]:
REPO_URL = 'https://github.com/Jx-study/aicup-2026-go.git'
COMMIT = 'origin/main'  # 重現特定版本時填 commit hash
MODE = 'pub2_smoke'
BATCH = 500
FAST = True
AUTO_UNASSIGN = True

HUMAN = 'b18c384nbt-humanv0.bin.gz'
MAIN = 'kata1-b28c512nbt-s13255194368-d5935380940.bin.gz'
P3 = ['--profiles', 'rank_11k', 'rank_1d', 'rank_6d']
TRAIN_CSV = ['--csv', '/content/training/train_*.csv']
HIST = ['--csv', '/content/games.csv', '--model', HUMAN, '--human-model', HUMAN, *P3, '--no-eval',
        '--pre-root-history', '--move-stats']
PUB2 = [*TRAIN_CSV, '--manifest', '/content/player_hsl_pub2_manifest.csv', *P3, '--pre-root-history',
        '--move-stats', '--no-eval']
OPP = ['--side', 'opponent']
VAL648 = [*TRAIN_CSV, '--players-file', '/content/val-players.csv', '--games-per-player', '20', *P3, '--no-eval']
VALNEW = [*TRAIN_CSV, '--players-file', '/content/val-players-new-v180000.csv', '--games-per-player', '20', *P3,
          '--no-eval']
POOL = [*TRAIN_CSV, '--players-file', '/content/train-pool-v180000.csv', '--games-per-player', '20', *P3, '--no-eval']
TEST = [*TRAIN_CSV, '--test-dir', '/content/tests', *P3, '--no-eval']
P3_2D = ['--profiles', 'rank_11k', 'rank_1d', 'rank_2d']
THREADS = ['--override-config', 'numAnalysisThreads=32,numSearchThreadsPerAnalysisThread=1']
TEST_HIST = ['--test-dir', '/content/tests', *HIST[2:]]
HA = 'humansl-analysis/'
RUNS = {  # MODE: (參數, Drive 上的輸出檔, 總局數)；總局數 None 時由輸入算出
    'trainpool_hist': (HIST, HA + 'hist6d-trainpool/trainpool-p3-hist.jsonl', None),
    'val648_hist': (HIST, HA + 'hist6d-val648/val648-p3-hist.jsonl', None),
    'newdev_hist': (HIST, HA + 'hist6d-newdev/newdev-p3-hist.jsonl', None),
    'pub2_smoke': (PUB2, 'player-trained-models/hsl-cache/pub2-p3-hist-smoke.jsonl', 100),
    'pub2': (PUB2, 'player-trained-models/hsl-cache/pub2-p3-hist.jsonl', 14200),
    'trainpool_hist_opp': (HIST + OPP, HA + 'hist6d-trainpool/trainpool-p3-opp-hist.jsonl', None),
    'val648_hist_opp': (HIST + OPP, HA + 'hist6d-val648/val648-p3-opp-hist.jsonl', None),
    'newdev_hist_opp': (HIST + OPP, HA + 'hist6d-newdev/newdev-p3-opp-hist.jsonl', None),
    'pub2_opp': (PUB2 + OPP, 'player-trained-models/hsl-cache/pub2-p3-hist-opp.jsonl', 14200),
    'val648': (VAL648, HA + 'val648-p3.jsonl', 12960),
    'val648_opp': (VAL648 + OPP, HA + 'val648-p3-opp.jsonl', 12960),
    'valnew': (VALNEW, HA + 'val-new-v180000-p3.jsonl', None),
    'valnew_opp': (VALNEW + OPP, HA + 'val-new-v180000-p3-opp.jsonl', None),
    'trainpool': (POOL, HA + 'train-pool-v180000-p3.jsonl', None),
    'trainpool_opp': (POOL + OPP, HA + 'train-pool-v180000-p3-opp.jsonl', None),
    'test': (TEST, HA + 'test-p3.jsonl', 10000),
    'test_opp': (TEST + OPP, HA + 'test-p3-opp.jsonl', 10000),
    'pilot': ([*TRAIN_CSV, '--players-per-rank', '1', '--games-per-player', '10'], HA + 'pilot-100-hd-v1.jsonl', 100),
    'visits': ([*TRAIN_CSV, '--players-per-rank', '1', '--games-per-player', '10', '--eval-visits', '16'],
               HA + 'pilot-100-hd-v16.jsonl', 100),
    'main': ([*TRAIN_CSV, '--players-per-rank', '30', '--games-per-player', '20'], HA + 'pilot-6000-hd.jsonl', 6000),
    'test_hist': (TEST_HIST, HA + 'hist6d-test/test-p3-hist.jsonl', 10000),
    'test_hist_opp': (TEST_HIST + OPP, HA + 'hist6d-test/test-p3-opp-hist.jsonl', 10000),
}
for run in ('val648', 'train_p1', 'train_p2', 'train_p3', 'holdout', 'train_p4', 'train_p4b', 'test'):
    src = ['--test-dir', '/content/tests'] if run == 'test' else ['--csv', '/content/games.csv']
    loss = ['--no-eval'] if run.startswith('train_p4') else ['--eval-visits', '1']
    args = [*src, '--model', MAIN, '--human-model', HUMAN, *loss, *P3_2D, '--pre-root-history', '--move-stats', *THREADS]
    base = HA + 'hist2d-%s/%s-p3' % (run, run)
    total = 10000 if run == 'test' else None
    RUNS[run + '_2d'] = (args, base + '-hist.jsonl', total)
    RUNS[run + '_2d_opp'] = (args + OPP, base + '-opp-hist.jsonl', total)
FAST_MODES = ('val648_opp', 'valnew_opp', 'test', 'test_opp')
FAST_ARGS = ['--model', HUMAN, '--group', '32', '--override-config',
             'numAnalysisThreads=128,numSearchThreadsPerAnalysisThread=1,nnMaxBatchSize=128']
VAL_PLAYERS_HINT = 'python -m aicup.val_players --val-size 180000 --out-dir humansl-analysis'
if MODE not in RUNS:
    raise SystemExit('MODE 要是 %s 之一' % ', '.join(RUNS))
ARGS, OUT, TOTAL = RUNS[MODE]
USE_FAST = FAST and MODE in FAST_MODES
USE_CTRL = '_2d' in MODE  # 2d MODE 先跑對照組

COPIES = []  # (Drive 上的路徑, /content 下的路徑, 產生方式)
if '/content/games.csv' in ARGS:
    run_dir = OUT.split('/')[1]
    prefix, run_name = run_dir.split('-', 1)
    flag = ' --2d' if prefix == 'hist2d' else ''
    COPIES.append((HA + run_dir + '/games.csv', '/content/games.csv',
                   'python -m aicup.hsl_games%s --run %s' % (flag, run_name)))
if MODE.startswith('pub2'):
    COPIES.append(('player-trained-models/hsl-cache/player_hsl_pub2_manifest.csv',
                   '/content/player_hsl_pub2_manifest.csv', 'python -m aicup.prep_player_hsl --tests-dir tests'))
for name in ('val-players.csv', 'val-players-new-v180000.csv', 'train-pool-v180000.csv'):
    if '/content/' + name in ARGS or (USE_FAST and name == 'val-players.csv'):
        COPIES.append((HA + name, '/content/' + name, VAL_PLAYERS_HINT))
if '--test-dir' in ARGS:
    for split in ('public', 'private'):
        name = 'rank_prediction_test_%s.csv' % split
        COPIES.append(('tests/' + name, '/content/tests/' + name, 'AI CUP 競賽平台'))
if USE_FAST or USE_CTRL:
    COPIES.append((HA + 'val648-p3.jsonl', '/content/val648-p3.jsonl', '這份 notebook 的 MODE val648'))
if USE_CTRL:
    COPIES.append((HA + 'hist2d-val648/games.csv', '/content/ctrl_games.csv',
                   'python -m aicup.hsl_games --2d --run val648'))
# 沒指定 --model 時 analyze_humansl 用主網路；FAST 比對失敗時也會改回主網路
NEED_MAIN = '--model' not in ARGS or MAIN in ARGS or USE_CTRL
READS_TRAIN = USE_FAST or (ARGS[:2] == TRAIN_CSV and '--manifest' not in ARGS and '--test-dir' not in ARGS)
CMD = ['python', '-m', 'aicup.analyze_humansl', '--model-dir', '/content/models', *ARGS]
print(' '.join(CMD), '--limit <n> --out /content/' + OUT.rsplit('/', 1)[1])

## 取得程式

要看到：Drive 掛載完成、最新的 commit、GPU 型號。

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)
from aicup.colab_nb import DRIVE_ROOT, check_commit, link_drive, need, run

GIT_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()

!git log --oneline -1
!nvidia-smi -L

## 準備資料

要看到：需要的檔案都在、複製到 `/content` 的檔案清單、`katago` 與 `katago-bin`、網路檔與 `analysis_example.cfg`。Drive 上已有結果時會複製回 `/content`，接續分析。

In [ ]:
import glob, shutil

need([(os.path.join(DRIVE_ROOT, src), how) for src, _, how in COPIES]
     + ([(os.path.join(DRIVE_ROOT, 'training-v1.1/training'), '官方訓練集 v1.1，見 README')] if READS_TRAIN else []))
for src, dst, _ in COPIES:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(os.path.join(DRIVE_ROOT, src), dst)
    print('複製', src, '->', dst)
if READS_TRAIN:
    os.makedirs('/content/training', exist_ok=True)
    for p in sorted(glob.glob(os.path.join(DRIVE_ROOT, 'training-v1.1/training/train_*.csv'))):
        shutil.copy(p, '/content/training/')
    print('訓練 CSV', len(os.listdir('/content/training')), '個')

os.environ['NEED_MAIN'] = '1' if NEED_MAIN else '0'
LOCAL = '/content/' + os.path.basename(OUT)
BACKUP = os.path.join(DRIVE_ROOT, os.path.dirname(OUT))
os.makedirs(BACKUP, exist_ok=True)
if os.path.exists(os.path.join(BACKUP, os.path.basename(OUT))) and not os.path.exists(LOCAL):
    shutil.copy(os.path.join(BACKUP, os.path.basename(OUT)), LOCAL)
    print('接續', OUT)
if TOTAL is None:
    import pandas as pd
    src = '/content/games.csv' if '/content/games.csv' in ARGS else ARGS[ARGS.index('--players-file') + 1]
    rows = len(pd.read_csv(src))
    TOTAL = rows if src == '/content/games.csv' else 20 * rows  # 玩家清單每人 20 局
    print('總局數', TOTAL)

In [ ]:
%%bash
mkdir -p /content/models /content/katago
cd /content/katago
ZIP=katago-v1.18.2-cuda12.8-cudnn9.8.0-linux-x64.zip
if [ ! -f katago-bin ]; then
    wget -q https://github.com/lightvector/KataGo/releases/download/v1.18.2/$ZIP && unzip -qjo $ZIP
    mv katago katago-bin
fi
# 包一層 script，讓 katago 找得到 Colab 上的 CUDA 函式庫
cat > katago << 'EOF'
#!/bin/bash
CUDA_PATHS=$(python3 -c "import glob; print(':'.join(glob.glob('/usr/local/cuda*/lib64') + glob.glob('/usr/local/lib/python3*/dist-packages/nvidia/*/lib')))")
export LD_LIBRARY_PATH=$CUDA_PATHS:/usr/local/cuda/lib64:$LD_LIBRARY_PATH
exec /content/katago/katago-bin "$@"
EOF
chmod +x katago katago-bin
cd /content/models
[ -f b18c384nbt-humanv0.bin.gz ] || wget -q https://github.com/lightvector/KataGo/releases/download/v1.15.0/b18c384nbt-humanv0.bin.gz
MAIN=kata1-b28c512nbt-s13255194368-d5935380940.bin.gz
if [ "$NEED_MAIN" = 1 ] && [ ! -f $MAIN ]; then
    wget -q https://media.katagotraining.org/uploaded/networks/models/kata1/$MAIN
fi
cd /content/katago
ls -la /content/models /content/katago/katago analysis_example.cfg

## 快速設定檢查

只在 `FAST` 用到的 MODE 執行。用快速設定重跑 val648 的前 40 局，和 `val648-p3.jsonl` 比對每個 profile 的每手平均 log 機率。比對一律評目標玩家的手，快速設定只換主模型與批次。40 局都要在參考檔中，且最大差小於 0.01 才採用。最後一行印出分析要用的指令。

In [ ]:
import json, time
import numpy as np

if USE_FAST:
    N = 40
    CHECK = '/content/fastcheck.jsonl'
    if os.path.exists(CHECK):
        os.remove(CHECK)
    t = time.time()
    run(['python', '-m', 'aicup.analyze_humansl', *TRAIN_CSV, '--model-dir', '/content/models',
         '--players-file', '/content/val-players.csv', '--games-per-player', '20', *P3, '--no-eval',
         '--limit', str(N), *FAST_ARGS, '--out', CHECK])
    sec = (time.time() - t) / N

    def mean_logp(probs):
        p = np.array(probs, dtype=float)
        return np.nanmean(np.log(np.clip(p, 1e-6, None)), axis=1)  # 同 fit_humansl 的特徵

    ref = {}
    for line in open('/content/val648-p3.jsonl'):
        r = json.loads(line)
        ref[r['key']] = mean_logp(r['probs'])
    rows = [json.loads(l) for l in open(CHECK)] if os.path.exists(CHECK) else []
    rows = [r for r in rows if r['key'] in ref]
    diff = max((np.max(np.abs(mean_logp(r['probs']) - ref[r['key']])) for r in rows), default=np.inf)
    print('快速設定：%d/%d 局可比對，含啟動每局 %.2f 秒，平均 log 機率的最大差 %.2g' % (len(rows), N, sec, diff))
    if len(rows) == N and diff < 0.01:
        CMD = CMD + FAST_ARGS
        print('採用快速設定')
    else:
        print('不採用快速設定，改用原設定')
print(' '.join(CMD))

## 對照組檢查

只在 2d MODE 執行。用 2d 的設定跑 val648 的前 40 局（不帶 `--pre-root-history`），和 `val648-p3.jsonl` 比對 `rank_11k`、`rank_1d` 的每手平均 log 機率。40 局都要在參考檔中，且中位數絕對差不超過 0.03 才繼續，否則停止：環境和原本的執行不一致。

In [ ]:
import json
import numpy as np

if USE_CTRL:
    N = 40
    CTRL = '/content/ctrl.jsonl'
    if os.path.exists(CTRL):
        os.remove(CTRL)
    run(['python', '-m', 'aicup.analyze_humansl', '--csv', '/content/ctrl_games.csv', '--model-dir', '/content/models',
         '--model', MAIN, '--human-model', HUMAN, *P3_2D, '--eval-visits', '1', *THREADS, '--limit', str(N),
         '--out', CTRL])

    def mean_logp(probs):
        p = np.array(probs, dtype=float)
        return np.nanmean(np.log(np.clip(p, 1e-6, None)), axis=1)

    ref = {}
    for line in open('/content/val648-p3.jsonl'):
        r = json.loads(line)
        ref[r['key']] = mean_logp(r['probs'])[:2]
        if len(ref) == N:
            break
    rows = [json.loads(l) for l in open(CTRL)]
    rows = [r for r in rows if r['key'] in ref]
    diffs = np.concatenate([np.abs(mean_logp(r['probs'])[:2] - ref[r['key']]) for r in rows]) if rows else np.array([np.inf])
    print('對照組 %d/%d 局，11k、1d 中位數絕對差 %.4g，最大差 %.4g' % (len(rows), N, np.median(diffs), diffs.max()))
    if len(rows) < N or np.median(diffs) > 0.03:
        raise SystemExit('對照組不一致，停止')

## 執行

每批結束會印出 Drive 上的行數，數字增加代表備份成功。

In [ ]:
LOG = '/content/%s.log' % MODE
for n in sorted({*range(BATCH, TOTAL, BATCH), TOTAL}):
    run([*CMD, '--limit', str(n), '--out', LOCAL], LOG)
    shutil.copy(LOCAL, BACKUP)
    shutil.copy(LOG, BACKUP)
    print('Drive 上 %d 行' % sum(1 for _ in open(os.path.join(BACKUP, os.path.basename(OUT)))))

## 檢查

要看到：行數等於總局數、沒有重複 key、`side` 與 `profiles` 符合 MODE、Drive 上的行數和本機相同。

In [ ]:
rows = [json.loads(l) for l in open(LOCAL)]
keys = [r['key'] for r in rows]
print('已完成 %d / %d 局，重複 key %d' % (len(rows), TOTAL, len(keys) - len(set(keys))))
print('side：', sorted({r.get('side', 'target') for r in rows}))
print('profiles：', sorted({tuple(r.get('profiles', ())) for r in rows}))
print('pre_root_history：', sorted({bool(r.get('pre_root_history', False)) for r in rows}))
n_drive = sum(1 for _ in open(os.path.join(BACKUP, os.path.basename(OUT))))
print('Drive：%s，%d 行，和本機相同：%s' % (OUT, n_drive, n_drive == len(rows)))

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣運算單元。

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()  # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')